# 32

Combine multiple CSV files into a single DataFrame, extracting city and state from the filename:

In [72]:
import pandas as pd
import numpy as np
import glob
import os

In [73]:
all_dfs = []

for one_filename in glob.glob('./data/*,*.csv'):
    print(f'Loading {one_filename}...')

    # Use os.path.basename to get just the filename without the directory path
    basename = os.path.basename(one_filename)

    # Extract city and state from the basename
    city, state = basename.removesuffix('.csv').split(',')

    one_df = (
        pd
        .read_csv(one_filename,
                  usecols=[0, 1, 2],
                  names=['date_time',
                         'max_temp',
                         'min_temp'],
                  header=0)
        .assign(city=city.replace('+', ' ').title(),
                state=state.upper())
    )

    all_dfs.append(one_df)

df = pd.concat(all_dfs)

Loading ./data\albany,ny.csv...
Loading ./data\boston,ma.csv...
Loading ./data\chicago,il.csv...
Loading ./data\los+angeles,ca.csv...
Loading ./data\new+york,ny.csv...
Loading ./data\san+francisco,ca.csv...
Loading ./data\springfield,il.csv...
Loading ./data\springfield,ma.csv...


Does the data for each city and state start and end at (roughly) the same time?

In [74]:
df.groupby(['state', 'city'])['date_time'].min().sort_values()

state  city         
CA     Los Angeles      2018-12-11 00:00:00
       San Francisco    2018-12-11 00:00:00
IL     Chicago          2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
MA     Boston           2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
NY     Albany           2018-12-11 00:00:00
       New York         2018-12-11 00:00:00
Name: date_time, dtype: object

In [75]:
df.groupby(['state', 'city'])['date_time'].max().sort_values()

state  city         
CA     Los Angeles      2019-03-11 21:00:00
       San Francisco    2019-03-11 21:00:00
IL     Chicago          2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
MA     Boston           2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
NY     Albany           2019-03-11 21:00:00
       New York         2019-03-11 21:00:00
Name: date_time, dtype: object

In [76]:
df.groupby(['state', 'city'])['date_time'].agg(['min', 'max'])

min                  max
state city                                                   
CA    Los Angeles    2018-12-11 00:00:00  2019-03-11 21:00:00
      San Francisco  2018-12-11 00:00:00  2019-03-11 21:00:00
IL    Chicago        2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
MA    Boston         2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
NY    Albany         2018-12-11 00:00:00  2019-03-11 21:00:00
      New York       2018-12-11 00:00:00  2019-03-11 21:00:00

What is the lowest minimum temperature recorded for each city in the data set?

In [77]:
df.groupby(['state', 'city'])['min_temp'].min()

state  city         
CA     Los Angeles       4
       San Francisco     3
IL     Chicago         -28
       Springfield     -25
MA     Boston          -14
       Springfield     -20
NY     Albany          -19
       New York        -14
Name: min_temp, dtype: int64

What is the highest maximum temperature recorded in each state in the data set?

In [78]:
df.groupby('state')['max_temp'].max()

state
CA    23
IL    16
MA    17
NY    15
Name: max_temp, dtype: int64

Run "describe" on the minimum and maximum temperature for each state-city combination

In [79]:
df.groupby(['state', 'city'])[['max_temp', 'min_temp']].apply(pd.DataFrame.describe)

max_temp    min_temp
state city                                     
CA    Los Angeles count  728.000000  728.000000
                  mean    17.054945   10.637363
                  std      2.708640    2.705200
                  min     12.000000    4.000000
                  25%     15.000000    9.000000
...                             ...         ...
NY    New York    min    -12.000000  -14.000000
                  25%      2.000000   -4.000000
                  50%      4.000000    0.000000
                  75%      7.000000    2.000000
                  max     15.000000   12.000000

[64 rows x 2 columns]

What is the average difference in temperature (i.e., max - min) for each of the cities in our data set?

In [80]:
df.groupby(['state', 'city'])[['min_temp', 'max_temp']].apply(lambda g: np.mean(g.max() - g.min()) )

state  city         
CA     Los Angeles      12.0
       San Francisco     8.0
IL     Chicago          34.0
       Springfield      35.5
MA     Boston           26.0
       Springfield      28.5
NY     Albany           26.5
       New York         26.5
dtype: float64

# 33

Read in the scores file (`sat-scores.csv`). This time, you want the following columns: `Year`, `State.Code`, `Total.Math`, `Family Income.Less than 20k.Math`,
`Family Income.Between 20-40k.Math`, `Family Income.Between 40-60k.Math`,
`Family Income.Between 60-80k.Math`, `Family Income.Between 80-100k.Math`,
and `Family Income.More than 100k.Math`.

In [81]:
filename = './data/sat-scores.csv'

df = pd.read_csv(filename,
                usecols=['Year', 'State.Code', 'Total.Math',
                         'Family Income.Less than 20k.Math',
                         'Family Income.Between 20-40k.Math',
                         'Family Income.Between 40-60k.Math',
                         'Family Income.Between 60-80k.Math',
                         'Family Income.Between 80-100k.Math',
                         'Family Income.More than 100k.Math'])
df.head()

,Year,State.Code,Total.Math,Family Income.Between 20-40k.Math,Family Income.Between 40-60k.Math,Family Income.Between 60-80k.Math,Family Income.Between 80-100k.Math,Family Income.Less than 20k.Math,Family Income.More than 100k.Math
0,2005,AL,559,513,539,550,566,462,588
1,2005,AK,519,492,517,513,528,464,541
2,2005,AZ,530,498,520,524,534,485,554
3,2005,AR,552,513,543,553,570,489,572
4,2005,CA,522,477,506,521,535,451,566


Rename the income-related column names to something shorter:
`income<20k`, `20k<income<40k`, `40k<income<60k`, `60k<income<80k`, `80k<income<100k`, and `income>100k`.

In [82]:
df.columns = ['Year', 'State.Code', 'Total.Math',
              'income<20k',
              '20k<income<40k',
              '40k<income<60k',
              '60k<income<80k',
              '80k<income<100k',
              'income>100k',
              ]

Find the average SAT math score for each income level, grouped and then
sorted by year

In [83]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .round(3)
)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
income<20k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20k<income<40k,0.070,0.041,0.050,0.046,0.044,0.046,0.037,0.041,0.043,0.035,0.045
40k<income<60k,0.026,0.021,0.026,0.003,0.005,0.023,0.030,0.026,0.017,0.024,0.026
60k<income<80k,0.024,0.029,0.023,0.015,0.021,0.025,0.025,0.024,0.033,0.030,0.028
80k<income<100k,-0.221,-0.162,-0.161,-0.142,-0.147,-0.129,-0.150,-0.148,-0.127,-0.154,-0.174
income>100k,0.338,0.242,0.234,0.180,0.215,0.193,0.223,0.215,0.185,0.209,0.259


Find the average SAT math score for each income level, grouped and then
sorted by year

In [84]:
df.groupby('Year').mean(numeric_only=True).sort_index().round(3)

,Total.Math,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,,
2005,535.654,488.654,522.673,536.077,548.942,427.596,572.173
2006,537.481,502.923,523.769,534.904,550.462,461.019,572.519
2007,535.340,494.849,519.491,533.189,545.698,457.925,565.170
2008,535.981,523.623,547.472,549.189,557.642,478.642,564.566
2009,540.804,527.824,550.980,553.941,565.333,482.059,585.784
2010,540.843,499.275,522.000,534.235,547.627,477.039,569.275
2011,533.226,494.887,513.415,528.660,541.849,460.453,563.245
2012,533.604,492.057,512.453,525.774,538.302,458.774,557.321
2013,532.623,490.132,511.377,520.321,537.396,469.358,556.340


For each year in the data set, determine how much better each income group
did, on average, than the next-poorer group of students.

In [85]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .T
    .mean()
    .sort_values(ascending=False)
    .head()
    .round(3)
)

income>100k        0.227
20k<income<40k     0.045
60k<income<80k     0.025
40k<income<60k     0.021
80k<income<100k   -0.156
dtype: float64

Which income levels consistently (i.e., across all years) do worse than the next-poorest group?

In [86]:
change = (
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .pct_change(axis='columns')
)

change[change < 0].dropna()

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,


Calculate descriptive statistics for all the changes in income brackets

In [87]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()
change.T.describe().round(3)

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
count,0.0,11.000,11.000,11.000,11.000,11.000
mean,NaN,0.045,0.021,0.025,-0.156,0.227
std,NaN,0.009,0.009,0.005,0.026,0.044
min,NaN,0.035,0.003,0.015,-0.221,0.180
25%,NaN,0.041,0.019,0.024,-0.162,0.201
50%,NaN,0.044,0.024,0.025,-0.150,0.215
75%,NaN,0.046,0.026,0.029,-0.144,0.238
max,NaN,0.070,0.030,0.033,-0.127,0.338


Which five states have the greatest gap in SAT math scores between the richest
and poorest students?

In [88]:
df['rich_poor_diff'] = df['income>100k'] - df['income<20k']
df.groupby('State.Code')['rich_poor_diff'].mean().sort_values(ascending=False).head().round(2)

State.Code
DC    182.00
MD     95.18
CT     95.00
DE     91.55
NJ     86.00
Name: rich_poor_diff, dtype: float64

Perform the same analysis on verbal SAT scores

In [89]:
df = pd.read_csv(filename,
                usecols=['Year', 'State.Code', 'Total.Verbal',
                         'Family Income.Less than 20k.Verbal',
                         'Family Income.Between 20-40k.Verbal',
                         'Family Income.Between 40-60k.Verbal',
                         'Family Income.Between 60-80k.Verbal',
                         'Family Income.Between 80-100k.Verbal',
                         'Family Income.More than 100k.Verbal'])

df.columns = ['Year', 'State.Code', 'Total.Verbal',
                      'income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k',
                      ]

df.head()

,Year,State.Code,Total.Verbal,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
0,2005,AL,567,527,551,564,577,474,590
1,2005,AK,523,500,522,519,534,467,544
2,2005,AZ,526,495,518,523,533,474,546
3,2005,AR,563,526,555,570,580,486,589
4,2005,CA,504,458,494,511,525,421,551


In [90]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .round(3)
)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
income<20k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20k<income<40k,0.048,0.043,0.052,0.040,0.038,0.046,0.043,0.044,0.047,0.032,0.045
40k<income<60k,0.022,0.018,0.021,0.003,0.006,0.023,0.029,0.026,0.018,0.035,0.027
60k<income<80k,0.023,0.022,0.018,0.014,0.023,0.025,0.017,0.015,0.026,0.018,0.021
80k<income<100k,-0.165,-0.170,-0.167,-0.141,-0.152,-0.143,-0.160,-0.156,-0.139,-0.163,-0.184
income>100k,0.240,0.245,0.238,0.174,0.209,0.207,0.238,0.222,0.199,0.214,0.269


In [91]:
df.groupby('Year').mean(numeric_only=True).sort_index().round(2)

,Total.Verbal,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,,
2005,534.27,501.10,524.96,536.25,548.40,457.83,567.52
2006,531.73,502.15,523.96,533.33,544.87,452.46,563.27
2007,531.53,496.58,522.43,533.17,542.81,452.40,559.98
2008,530.72,522.92,543.70,545.28,552.91,474.83,557.47
2009,535.14,526.84,547.00,550.02,562.57,476.92,576.61
2010,535.86,497.65,520.57,532.61,545.82,467.65,564.39
2011,528.81,493.21,514.17,529.25,538.11,451.91,559.38
2012,527.36,491.11,512.89,526.23,534.32,450.85,550.83
2013,528.32,490.06,513.02,522.42,536.08,461.64,553.49


In [92]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()
change.T.describe().round(3)

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
count,0.0,11.000,11.000,11.000,11.000,11.000
mean,NaN,0.043,0.021,0.020,-0.158,0.223
std,NaN,0.005,0.010,0.004,0.014,0.026
min,NaN,0.032,0.003,0.014,-0.184,0.174
25%,NaN,0.041,0.018,0.017,-0.166,0.208
50%,NaN,0.044,0.022,0.021,-0.160,0.222
75%,NaN,0.046,0.027,0.023,-0.148,0.239
max,NaN,0.052,0.035,0.026,-0.139,0.269


In [93]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()

change[change <= 0].dropna().round(3)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
80k<income<100k,-0.165,-0.17,-0.167,-0.141,-0.152,-0.143,-0.16,-0.156,-0.139,-0.163,-0.184


In [94]:
df['rich_poor_diff'] = df['income>100k'] - df['income<20k']
df.groupby('State.Code')['rich_poor_diff'].mean().sort_values(ascending=False).head().round(2)

State.Code
DC    182.45
MD     89.09
DE     86.45
CT     84.55
CA     84.09
Name: rich_poor_diff, dtype: float64